# Module 4: Open Quantum Systems, Dissipation & Trajectories
## Notebook 4.2: The Lindblad Master Equation & Dynamical Semigroups

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/qutip/computational-quantum-optics/blob/main/module_4_open_systems_dissipation/03_lindblad_master_equation.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-Repository-blue?logo=github)](https://github.com/qutip/computational-quantum-optics)
[![Python 3.10+](https://img.shields.io/badge/Python-3.10%2B-brightgreen?logo=python)](https://www.python.org/)

---

### Computational Quantum Optics Curriculum
* **Module 0:** Mathematical & Quantum Foundations
* **Module 1:** Single-Mode Quantum States of Light
* **Module 2:** Phase Space Quasiprobability Distributions
* **Module 3:** Light-Matter Interaction & Cavity QED
* **Module 4:** **Open Quantum Systems, Dissipation & Trajectories**
  * **4.1** Density Matrix, Purity & Entanglement Entropy
  * **4.2 The Lindblad Master Equation & Dynamical Semigroups** $\leftarrow$ *(You are here)*
  * **4.3** Cavity Decay, Photon Loss & Thermal Reservoirs
  * **4.4** The Monte Carlo Wavefunction (Quantum Jump) Method
  * **4.5** Quantum Trajectories & Homodyne Detection Unraveling
  * **4.6** Input-Output Theory & Cavity Scattering
  * **4.7** Laser Theory, Micromaser & Trapping States

---

### Learning Objectives
1. **Microscopic Derivation of Master Equations:** Trace the approximations connecting microscopic system-reservoir Hamiltonians to Markovian master equations (Born, Markov, and Secular approximations).
2. **The GKSL Theorem & Dynamical Semigroups:** Understand why the Gorini-Kossakowski-Sudarshan-Lindblad form is the unique generator preserving Hermiticity, unit trace, and complete positivity for all time.
3. **Canonical Quantum Dissipative Channels:** Formulate jump operators for spontaneous radiative decay ($\hat{\sigma}_-$), thermal excitation ($\hat{\sigma}_+$), and pure dephasing ($\hat{\sigma}_z$).
4. **Liouvillian Superoperators & Vectorization:** Transform the density matrix into a $d^2$-dimensional column vector, construct the Liouvillian supermatrix $\mathcal{L}$, and solve for steady states via null-space diagonalization.
5. **The Dissipative Gap & Relaxation Timescales:** Connect the real parts of Liouvillian eigenvalues to decay rates and identify the asymptotic relaxation timescale set by the Liouvillian spectral gap $\Delta_{\text{gap}}$.


In [ ]:
# Environment Setup: Silent Colab detection & high-DPI publication styling
import os
import sys

# Configure headless Agg backend to ensure bulletproof plotting across all environments
if "MPLBACKEND" not in os.environ:
    os.environ["MPLBACKEND"] = "Agg"

import math
import numpy as np
import scipy.linalg as la
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

# Configure publication matplotlib styling
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.titlesize": 14,
    "lines.linewidth": 2.0,
    "axes.grid": True,
    "grid.alpha": 0.35,
    "grid.linestyle": "--"
})

try:
    import qutip as qt
    HAS_QUTIP = True
    print("QuTiP successfully imported.")
except ImportError:
    HAS_QUTIP = False
    print("QuTiP not found. Using native NumPy/SciPy implementations.")

try:
    import ipywidgets as widgets
    from IPython.display import display
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False
    print("ipywidgets not found. Interactive exploration will run in static fallback mode.")


---

## 1. Microscopic Foundations of Open Quantum Systems

A quantum system $S$ coupled to an uncontrolled macroscopic environment (reservoir / bath) $B$ is governed by the total Hamiltonian:
$$\hat{H}_{\text{tot}} = \hat{H}_S \otimes \hat{I}_B + \hat{I}_S \otimes \hat{H}_B + \hat{H}_I$$

The microscopic von Neumann equation for the total density operator is unitary:
$$\frac{d\hat{\rho}_{\text{tot}}}{dt} = -\frac{i}{\hbar} [\hat{H}_{\text{tot}}, \hat{\rho}_{\text{tot}}]$$

To obtain an autonomous equation of motion for the reduced density matrix $\hat{\rho}_S(t) = \mathrm{Tr}_B(\hat{\rho}_{\text{tot}}(t))$, three foundational physical approximations are made:
1. **Born Approximation (Weak Coupling):** The system-reservoir interaction is weak, and the reservoir has infinite degrees of freedom in thermal equilibrium. Correlations between system and bath remain negligible at all times:
   $$\hat{\rho}_{\text{tot}}(t) \approx \hat{\rho}_S(t) \otimes \hat{\rho}_B$$
2. **Markov Approximation (Short Memory Time):** The reservoir correlation time $\tau_B$ (the timescale over which bath fluctuations decay) is infinitely faster than the characteristic system relaxation timescale $\tau_R$:
   $$\tau_B \ll \tau_R \implies \hat{\rho}_S(t - \tau) \approx \hat{\rho}_S(t)$$
   The system has no memory: its future evolution depends only on its present state.
3. **Secular (Rotating-Wave) Approximation:** Terms oscillating rapidly at frequency differences $|\omega - \omega'| \gg 1/\tau_R$ are averaged to zero over the coarse-grained dissipative timescale.

---

## 2. The GKSL Theorem & The Lindblad Form

In 1976, **Vittorio Gorini, Andrzej Kossakowski, E. C. George Sudarshan**, and independently **Göran Lindblad**, proved a mathematical theorem of monumental importance in quantum mechanics:

> **The GKSL Theorem:**
> *A linear, time-homogeneous quantum dynamical map $\hat{\rho}(t) = \mathcal{E}_t \hat{\rho}(0)$ forms a Markovian dynamical semigroup preserving trace ($\mathrm{Tr}\hat{\rho} = 1$), Hermiticity ($\hat{\rho}^\dagger = \hat{\rho}$), and Complete Positivity ($\mathcal{E}_t \ge 0$) for all $t \ge 0$ if and only if its generator $\mathcal{L} = \frac{d\hat{\rho}}{dt}$ has the Lindblad form:*

$$\frac{d\hat{\rho}}{dt} = \mathcal{L}\hat{\rho} = -\frac{i}{\hbar} [\hat{H}, \hat{\rho}] + \sum_{k} \gamma_k \mathcal{D}[\hat{L}_k]\hat{\rho}$$
where $\hat{H}$ is the effective Hermitian system Hamiltonian (including Lamb shifts), $\hat{L}_k$ are the dimensionless **Lindblad jump operators**, and $\gamma_k \ge 0$ are the decay rates.

### The Lindblad Dissipator:
$$\mathcal{D}[\hat{L}]\hat{\rho} = \hat{L}\hat{\rho}\hat{L}^\dagger - \frac{1}{2} \hat{L}^\dagger\hat{L}\hat{\rho} - \frac{1}{2} \hat{\rho}\hat{L}^\dagger\hat{L} = \hat{L}\hat{\rho}\hat{L}^\dagger - \frac{1}{2} \{ \hat{L}^\dagger\hat{L}, \hat{\rho} \}$$

Physical anatomy of the dissipator:
* **The "Jump" Term ($\hat{L}\hat{\rho}\hat{L}^\dagger$):** Represents physical, discontinuous quantum jumps (e.g. emission of a photon into the environment).
* **The Anti-Commutator Term ($-\frac{1}{2}\{\hat{L}^\dagger\hat{L}, \hat{\rho}\}$):** Represents continuous non-Hermitian damping between jumps, strictly enforcing conservation of total probability $\mathrm{Tr}(\mathcal{D}[\hat{L}]\hat{\rho}) = 0$.

---

## 3. Superoperator Vectorization & Liouvillian Spectrum

A $d \times d$ density matrix $\hat{\rho}$ can be flattened into a $d^2$-dimensional column vector $|\rho\rangle\rangle = \mathrm{vec}(\hat{\rho})$ by stacking its columns.
Using the Kronecker product identity:
$$\mathrm{vec}(\hat{A}\hat{B}\hat{C}) = (\hat{C}^T \otimes \hat{A}) \mathrm{vec}(\hat{B})$$

The Lindblad master equation becomes a standard linear matrix differential equation:
$$\frac{d|\rho\rangle\rangle}{dt} = \hat{\mathcal{L}} |\rho\rangle\rangle$$
where the $d^2 \times d^2$ **Liouvillian superoperator matrix** is:
$$\hat{\mathcal{L}} = -i \left( \hat{I} \otimes \hat{H} - \hat{H}^T \otimes \hat{I} \right) + \sum_{k} \gamma_k \left[ \hat{L}_k^* \otimes \hat{L}_k - \frac{1}{2} \hat{I} \otimes (\hat{L}_k^\dagger \hat{L}_k) - \frac{1}{2} (\hat{L}_k^\dagger \hat{L}_k)^T \otimes \hat{I} \right]$$

### Spectral Properties of $\hat{\mathcal{L}}$:
1. **Eigenvalues:** $\hat{\mathcal{L}} |v_j\rangle\rangle = \lambda_j |v_j\rangle\rangle$.
   All eigenvalues lie in the non-positive half-plane: $\mathrm{Re}(\lambda_j) \le 0$.
2. **The Steady State ($\lambda_0 = 0$):**
   There is always at least one zero eigenvalue $\lambda_0 = 0$. Its eigenvector $|v_0\rangle\rangle$ reshapes into the unique stationary state $\hat{\rho}_{\text{ss}}$ satisfying $\mathcal{L}\hat{\rho}_{\text{ss}} = 0$.
3. **The Dissipative Spectral Gap:**
   $$\Delta_{\text{gap}} = \min_{j \ne 0} |\mathrm{Re}(\lambda_j)|$$
   The dissipative gap sets the **asymptotic relaxation rate**: any non-equilibrium state relaxes toward steady state as $\exp(-\Delta_{\text{gap}} t)$!


In [ ]:
# Implementation: Full Vectorized Lindblad Superoperator Solver & Eigenspectrum

class LindbladMasterSimulator:
    """
    Constructs the exact d^2 x d^2 vectorized Liouvillian superoperator
    and simulates open quantum dynamics for arbitrary Hamiltonians and jump operators.
    """
    def __init__(self, H, c_ops, rates=None):
        """
        Parameters:
            H: d x d Hermitian Hamiltonian matrix.
            c_ops: list of d x d collapse (jump) operators L_k.
            rates: list of decay rates gamma_k (defaults to 1.0 for each).
        """
        self.H = np.array(H, dtype=complex)
        self.d = self.H.shape[0]
        self.d2 = self.d * self.d
        
        self.c_ops = [np.array(op, dtype=complex) for op in c_ops]
        if rates is None:
            self.rates = [1.0] * len(self.c_ops)
        else:
            self.rates = [float(r) for r in rates]
            
        self._build_liouvillian()
        
    def _build_liouvillian(self):
        """Constructs the vectorized Liouvillian supermatrix L of shape (d^2, d^2)."""
        I_d = np.eye(self.d, dtype=complex)
        
        # Unitary part: -i * (I (x) H - H^T (x) I)
        L = -1j * (np.kron(I_d, self.H) - np.kron(self.H.T, I_d))
        
        # Dissipative part: sum_k gamma_k * [ L_k^* (x) L_k - 0.5 * I (x) L_k^\dagger L_k - 0.5 * (L_k^\dagger L_k)^T (x) I ]
        for L_k, rate in zip(self.c_ops, self.rates):
            L_dag_L = L_k.conj().T @ L_k
            term_jump = np.kron(L_k.conj(), L_k)
            term_damping = 0.5 * np.kron(I_d, L_dag_L) + 0.5 * np.kron(L_dag_L.T, I_d)
            L += rate * (term_jump - term_damping)
            
        self.L = L
        
    def steady_state(self):
        """Finds the steady-state density matrix satisfying L |rho_ss>> = 0."""
        evals, evecs = la.eig(self.L)
        # Find index with eigenvalue closest to zero
        idx_zero = np.argmin(np.abs(evals))
        rho_vec = evecs[:, idx_zero]
        
        # Un-vectorize (column-major)
        rho_ss = rho_vec.reshape((self.d, self.d), order='F')
        # Normalize trace and enforce Hermiticity
        rho_ss = 0.5 * (rho_ss + rho_ss.conj().T)
        rho_ss /= np.trace(rho_ss)
        return rho_ss
        
    def spectral_gap(self):
        """Calculates the dissipative gap: min_{j != 0} |Re(lambda_j)|."""
        evals = la.eigvals(self.L)
        real_parts = np.sort(np.real(evals))[::-1]  # Descending: [0, -gap, ...]
        # First non-zero element
        non_zeros = real_parts[np.abs(real_parts) > 1e-10]
        if len(non_zeros) == 0:
            return 0.0
        return abs(non_zeros[0])
        
    def evolve(self, rho0, t_arr):
        """Propagates rho0 under the Liouvillian via matrix exponential."""
        rho_vec0 = rho0.flatten(order='F')
        # Diagonalize L: L = V * D * V^-1
        evals, V = la.eig(self.L)
        V_inv = la.inv(V)
        
        c0 = V_inv @ rho_vec0
        
        # rho(t) = V @ exp(D * t) @ c0
        states = []
        for t in t_arr:
            exp_diag = np.exp(evals * t)
            vec_t = V @ (exp_diag * c0)
            rho_t = vec_t.reshape((self.d, self.d), order='F')
            # Hermitize and normalize trace
            rho_t = 0.5 * (rho_t + rho_t.conj().T)
            rho_t /= np.trace(rho_t)
            states.append(rho_t)
            
        return states

# Verification Benchmark: Damped Two-Level System (Spontaneous Decay + Detuned Drive)
# H = hbar*Omega/2 * sigma_x, jump = sigma_- with rate Gamma
sigma_x = np.array([[0, 1], [1, 0]], dtype=complex)
sigma_minus = np.array([[0, 0], [1, 0]], dtype=complex)

omega_drive = 2.0
gamma_decay = 1.0
sim_master = LindbladMasterSimulator(H=0.5 * omega_drive * sigma_x, c_ops=[sigma_minus], rates=[gamma_decay])

# Compute steady state
rho_ss_num = sim_master.steady_state()

# Analytical steady state for driven-dissipative two-level atom
s0 = 2.0 * (omega_drive / gamma_decay)**2
rho_ee_analytical = 0.5 * s0 / (1.0 + s0)

gap_val = sim_master.spectral_gap()

print("--- Lindblad Superoperator Benchmark ---")
print(f"Liouvillian Dimension:               {sim_master.d2} x {sim_master.d2}")
print(f"Dissipative Spectral Gap:            {gap_val:.4f} Gamma")
print(f"Numerical Steady-State rho_ee:       {np.real(rho_ss_num[0, 0]):.6f}")
print(f"Analytical Steady-State rho_ee:      {rho_ee_analytical:.6f}")
print(f"Discrepancy:                         {abs(np.real(rho_ss_num[0, 0]) - rho_ee_analytical):.3e}")

assert abs(np.real(rho_ss_num[0, 0]) - rho_ee_analytical) < 1e-12, "Steady state master equation benchmark failed!"
assert abs(np.trace(rho_ss_num) - 1.0) < 1e-12, "Unit trace check failed!"
print("Verification Passed: Vectorized Liouvillian yields exact analytical steady state to machine precision!")


In [ ]:
# Publication-Quality 4-Panel Visualization: Damped Rabi Dynamics, Channel Geometry, Thermal Steady State & Liouvillian Spectrum

fig = plt.figure(figsize=(16, 12))

# =============================================================================
# Panel 1: Damped Driven Dynamics: Excited Population & Coherences
# =============================================================================
ax1 = fig.add_subplot(2, 2, 1)

t_span = np.linspace(0.0, 10.0, 500)
# Start in ground state |g> = [0, 1]^T -> rho0 = [[0, 0], [0, 1]]
rho_init = np.array([[0.0, 0.0], [0.0, 1.0]], dtype=complex)
states_t = sim_master.evolve(rho_init, t_span)

rho_ee_t = [np.real(s[0, 0]) for s in states_t]
rho_ge_re = [np.real(s[1, 0]) for s in states_t]
rho_ge_im = [np.imag(s[1, 0]) for s in states_t]

ax1.plot(t_span, rho_ee_t, 'b-', lw=2.2, label=r"Excited Population $\rho_{ee}(t)$")
ax1.plot(t_span, rho_ge_im, 'r--', lw=1.8, label=r"Absorption Coherence $\mathrm{Im}(\rho_{ge})$")
ax1.plot(t_span, rho_ge_re, 'g-.', lw=1.8, label=r"Dispersion Coherence $\mathrm{Re}(\rho_{ge})$")
ax1.axhline(rho_ee_analytical, color='navy', linestyle=':', label=rf"Steady-State $\rho_{{ee}}^{{\mathrm{{ss}}}} = {rho_ee_analytical:.3f}$")

ax1.set_xlabel(r"Time ($\Gamma t$)")
ax1.set_ylabel("Density Matrix Component")
ax1.set_title(r"Lindblad Dynamics: Driven Two-Level Atom ($\Omega = 2\Gamma$)")
ax1.set_ylim(-0.55, 1.05)
ax1.legend(loc="upper right", frameon=True)

# =============================================================================
# Panel 2: Bloch Ball Dissipative Deformations (Pure Dephasing vs. Amplitude Damping)
# =============================================================================
ax2 = fig.add_subplot(2, 2, 2)

theta_c = np.linspace(0, 2*np.pi, 200)
# Original unit sphere cross section
ax2.plot(np.cos(theta_c), np.sin(theta_c), 'k-', lw=1.5, label="Initial Pure States ($r = 1$)")

# Amplitude damping deformed ellipsoid (shifts toward south pole w = -1)
# u(t) = u0 * exp(-gamma*t/2), w(t) = -1 + (w0 + 1)*exp(-gamma*t)
t_decay = 1.0
scale_u_amp = np.exp(-0.5 * t_decay)
scale_w_amp = np.exp(-1.0 * t_decay)
center_w_amp = -1.0 + np.exp(-1.0 * t_decay)
ax2.plot(scale_u_amp * np.cos(theta_c), center_w_amp + scale_w_amp * np.sin(theta_c),
         'r--', lw=2.2, label=rf"Amplitude Damping ($\Gamma t = {t_decay:.1f}$)")

# Pure dephasing deformed ellipsoid (contracts horizontally, vertical w invariant)
scale_u_deph = np.exp(-1.0 * t_decay)
ax2.plot(scale_u_deph * np.cos(theta_c), np.sin(theta_c),
         'b-.', lw=2.2, label=rf"Pure Dephasing ($\gamma_\phi t = {t_decay:.1f}$)")

ax2.plot([0], [-1], 'ro', markersize=8, label=r"Ground State $|g\rangle$")
ax2.plot([0], [0], 'ko', markersize=5)

ax2.set_xlim(-1.18, 1.18)
ax2.set_ylim(-1.18, 1.18)
ax2.set_aspect('equal')
ax2.set_xlabel(r"Bloch Component $u = \mathrm{Tr}(\hat{\rho}\hat{\sigma}_x)$")
ax2.set_ylabel(r"Bloch Component $w = \mathrm{Tr}(\hat{\rho}\hat{\sigma}_z)$")
ax2.set_title("Bloch Ball Deformations under Open Dissipative Channels")
ax2.legend(loc="upper right", fontsize=8.5, frameon=True)

# =============================================================================
# Panel 3: Thermal Reservoir Equilibrium vs. Mean Thermal Occupation n_th
# =============================================================================
ax3 = fig.add_subplot(2, 2, 3)

n_th_scan = np.linspace(0.0, 3.0, 100)
rho_ee_thermal = []

for nth in n_th_scan:
    # Jump operators: L_down = sqrt(Gamma * (1 + nth)) * sigma_minus
    #                 L_up   = sqrt(Gamma * nth) * sigma_plus
    rate_down = 1.0 * (1.0 + nth)
    rate_up   = 1.0 * nth
    sigma_plus = sigma_minus.T
    
    sim_th = LindbladMasterSimulator(H=np.zeros((2, 2), dtype=complex),
                                     c_ops=[sigma_minus, sigma_plus],
                                     rates=[rate_down, rate_up])
    rho_th = sim_th.steady_state()
    rho_ee_thermal.append(np.real(rho_th[0, 0]))

ax3.plot(n_th_scan, rho_ee_thermal, 'purple', lw=2.5, label=r"Steady-State $\rho_{ee}^{\mathrm{ss}}$")
# Analytical: rho_ee = nth / (2*nth + 1)
ax3.plot(n_th_scan, n_th_scan / (2.0 * n_th_scan + 1.0), 'k:', lw=1.8, label=r"Boltzmann Law $\frac{\bar{n}_{\mathrm{th}}}{2\bar{n}_{\mathrm{th}} + 1}$")
ax3.axhline(0.5, color='gray', linestyle='--', label=r"Infinite Temperature Limit $\rho_{ee} = 0.5$")

ax3.set_xlabel(r"Thermal Reservoir Photon Number $\bar{n}_{\mathrm{th}}$")
ax3.set_ylabel(r"Excited State Population $\rho_{ee}^{\mathrm{ss}}$")
ax3.set_title("Thermal Reservoir: Detailed Balance Equilibrium")
ax3.set_ylim(-0.02, 0.58)
ax3.legend(loc="lower right", frameon=True)

# =============================================================================
# Panel 4: Liouvillian Superoperator Spectrum in the Complex Plane
# =============================================================================
ax4 = fig.add_subplot(2, 2, 4)

# Calculate Liouvillian eigenvalues for driven-dissipative system
evals_L = la.eigvals(sim_master.L)

ax4.scatter(np.real(evals_L), np.imag(evals_L), color='crimson', s=60, zorder=5, label="Liouvillian Eigenvalues")
ax4.axvline(0.0, color='k', linestyle='-', lw=1.2)
ax4.axhline(0.0, color='k', linestyle=':', lw=0.8)

# Highlight zero eigenvalue (Steady State)
idx_z = np.argmin(np.abs(evals_L))
ax4.scatter([np.real(evals_L[idx_z])], [np.imag(evals_L[idx_z])], color='gold', s=140, edgecolors='k', zorder=6,
            label=r"Steady State $\lambda_0 = 0$")

# Mark spectral gap
gap = sim_master.spectral_gap()
ax4.axvline(-gap, color='dodgerblue', linestyle='--', lw=1.8, label=rf"Dissipative Gap $\Delta_{{\mathrm{{gap}}}} = {gap:.3f}\Gamma$")

ax4.set_xlabel(r"Decay Rate $\mathrm{Re}(\lambda) / \Gamma$ (Strictly $\leq 0$)")
ax4.set_ylabel(r"Oscillation Frequency $\mathrm{Im}(\lambda) / \Gamma$")
ax4.set_title(r"Liouvillian Spectrum in the Complex Plane")
ax4.legend(loc="upper left", frameon=True)

plt.tight_layout()
plt.show()


> **Physical Insight: Why Is Complete Positivity Non-Negotiable in Quantum Mechanics?**
>
> In classical probability, any linear transformation that maps a non-negative probability vector to another non-negative probability vector is a valid stochastic matrix (**Positivity**).
>
> In quantum mechanics, however, ordinary positivity is **not enough**. Consider the simple matrix transposition map:
> $$\mathcal{T}(\hat{\rho}) = \hat{\rho}^T$$
> For any isolated quantum system, $\mathcal{T}$ preserves Hermiticity, trace, and non-negativity of eigenvalues.
>
> However, suppose our system $S$ is entangled with an inert spectator qubit $A$ in a Bell state $|\Phi^+\rangle = \frac{|00\rangle + |11\rangle}{\sqrt{2}}$. Applying the trivial identity map to $A$ and the transpose map to $S$ creates the partial transpose:
> $$\hat{\rho}^{T_S} = (\hat{I}_A \otimes \mathcal{T})(\hat{\rho}_{AS})$$
> As shown in the Peres-Horodecki criterion, $\hat{\rho}^{T_S}$ develops **negative eigenvalues**! A negative eigenvalue implies a negative probability of finding the system in that state—a physical absurdity.
>
> A quantum map $\mathcal{E}$ is **Completely Positive (CP)** if and only if:
> $$\hat{I}_N \otimes \mathcal{E} \ge 0 \quad \forall N \in \mathbb{N}$$
> The GKSL Lindblad master equation is the **unique** differential equation that guarantees Complete Positivity for all time, ensuring that open quantum dynamics remain physically consistent even in the presence of external entangled ancillas.


In [ ]:
# Interactive Exploration: Master Equation Dynamics with Detuning & Dephasing

def plot_interactive_master(omega=2.0, delta=0.0, gamma_decay=1.0, gamma_deph=0.2):
    """
    Simulates the open dynamics of a driven two-level atom with spontaneous emission
    and pure dephasing, tracking the Liouvillian spectral gap.
    """
    # H = -delta/2 * sigma_z + omega/2 * sigma_x
    sz = np.array([[1, 0], [0, -1]], dtype=complex)
    sx = np.array([[0, 1], [1, 0]], dtype=complex)
    sm = np.array([[0, 0], [1, 0]], dtype=complex)
    
    H_sys = -0.5 * delta * sz + 0.5 * omega * sx
    c_ops = [sm, sz]
    rates = [gamma_decay, gamma_deph]
    
    sim = LindbladMasterSimulator(H=H_sys, c_ops=c_ops, rates=rates)
    gap = sim.spectral_gap()
    
    t_ev = np.linspace(0.0, 8.0, 500)
    rho_0 = np.array([[0, 0], [0, 1]], dtype=complex)
    states = sim.evolve(rho_0, t_ev)
    
    pe_ev = [np.real(s[0, 0]) for s in states]
    purity_ev = [np.real(np.trace(s @ s)) for s in states]
    
    fig, (ax_pop, ax_pur) = plt.subplots(1, 2, figsize=(14, 4.5))
    
    # Population
    ax_pop.plot(t_ev, pe_ev, 'b-', lw=2.2, label=r"Excited Population $\rho_{ee}(t)$")
    ax_pop.set_xlabel(r"Time ($\Gamma t$)")
    ax_pop.set_ylabel(r"$\rho_{ee}(t)$")
    ax_pop.set_title(rf"Lindblad Population Dynamics (Gap $\Delta={gap:.3f}\Gamma$)")
    ax_pop.set_ylim(-0.02, 1.05)
    ax_pop.legend(frameon=True)
    
    # Purity
    ax_pur.plot(t_ev, purity_ev, 'purple', lw=2.2, label=r"Purity $\gamma(t) = \mathrm{Tr}(\hat{\rho}^2)$")
    ax_pur.axhline(0.5, color='gray', linestyle='--', label="Max Mixed State 0.5")
    ax_pur.set_xlabel(r"Time ($\Gamma t$)")
    ax_pur.set_ylabel(r"Purity $\gamma(t)$")
    ax_pur.set_title("Loss of Purity & Mixed State Evolution")
    ax_pur.set_ylim(0.45, 1.05)
    ax_pur.legend(frameon=True)
    
    plt.tight_layout()
    plt.show()

if HAS_WIDGETS:
    widgets.interact(
        plot_interactive_master,
        omega=widgets.FloatSlider(value=2.0, min=0.0, max=6.0, step=0.5, description=r"$\Omega$:"),
        delta=widgets.FloatSlider(value=0.0, min=-3.0, max=3.0, step=0.5, description=r"$\Delta$:"),
        gamma_decay=widgets.FloatSlider(value=1.0, min=0.1, max=3.0, step=0.2, description=r"$\Gamma$:"),
        gamma_deph=widgets.FloatSlider(value=0.2, min=0.0, max=2.0, step=0.1, description=r"$\gamma_\phi$:")
    )
else:
    print("Static exploration demonstration (default parameters: Omega = 2.0, Delta = 0.0, Gamma = 1.0, dephasing = 0.2):")
    plot_interactive_master(omega=2.0, delta=0.0, gamma_decay=1.0, gamma_deph=0.2)


---

## 4. Self-Check Exercises & Guided Challenge

### Challenge 1: The Transverse Coherence Decay Rate
For an undriven two-level atom ($H = 0$) subject to spontaneous radiative decay at rate $\Gamma$ ($\hat{L}_1 = \hat{\sigma}_-$) and pure elastic dephasing at rate $\gamma_\phi$ ($\hat{L}_2 = \frac{1}{\sqrt{2}}\hat{\sigma}_z$):
$$\frac{d\hat{\rho}}{dt} = \Gamma \mathcal{D}[\hat{\sigma}_-]\hat{\rho} + \frac{\gamma_\phi}{2} \mathcal{D}[\hat{\sigma}_z]\hat{\rho}$$

Prove that the off-diagonal coherence decays exponentially as:
$$\rho_{10}(t) = \rho_{10}(0) \exp\left( - \left[ \frac{\Gamma}{2} + \gamma_\phi \right] t \right) \equiv \rho_{10}(0) \exp\left( -\frac{t}{T_2} \right)$$

<details>
<summary><b>Solution & Derivation</b> (Click to expand)</summary>

Evaluating the action of the dissipators on $\rho_{10} = \langle 1|\hat{\rho}|0\rangle$:
1. **Spontaneous emission dissipator:**
   $$\hat{\sigma}_- \hat{\rho} \hat{\sigma}_+ = \begin{pmatrix} 0 & 0 \\ 1 & 0 \end{pmatrix} \begin{pmatrix} \rho_{00} & \rho_{01} \\ \rho_{10} & \rho_{11} \end{pmatrix} \begin{pmatrix} 0 & 1 \\ 0 & 0 \end{pmatrix} = \begin{pmatrix} 0 & 0 \\ 0 & \rho_{00} \end{pmatrix}$$
   Notice that the jump term has zero off-diagonal elements!
   The damping term:
   $$\hat{\sigma}_+ \hat{\sigma}_- = |0\rangle\langle 0| = \begin{pmatrix} 1 & 0 \\ 0 & 0 \end{pmatrix}$$
   $$\frac{1}{2} \{ \hat{\sigma}_+ \hat{\sigma}_-, \hat{\rho} \} = \frac{1}{2} \begin{pmatrix} 2\rho_{00} & \rho_{01} \\ \rho_{10} & 0 \end{pmatrix}$$
   Therefore:
   $$\langle 1| \mathcal{D}[\hat{\sigma}_-]\hat{\rho} |0\rangle = 0 - \frac{1}{2}\rho_{10} = -\frac{1}{2}\rho_{10}$$

2. **Pure dephasing dissipator:**
   With $\hat{\sigma}_z = \begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix}$, $\hat{\sigma}_z^2 = \hat{I}$:
   $$\mathcal{D}[\hat{\sigma}_z]\hat{\rho} = \hat{\sigma}_z \hat{\rho} \hat{\sigma}_z - \hat{\rho} = \begin{pmatrix} \rho_{00} & -\rho_{01} \\ -\rho_{10} & \rho_{11} \end{pmatrix} - \begin{pmatrix} \rho_{00} & \rho_{01} \\ \rho_{10} & \rho_{11} \end{pmatrix} = \begin{pmatrix} 0 & -2\rho_{01} \\ -2\rho_{10} & 0 \end{pmatrix}$$
   Multiplying by $\gamma_\phi / 2$:
   $$\langle 1 | \frac{\gamma_\phi}{2} \mathcal{D}[\hat{\sigma}_z]\hat{\rho} |0\rangle = -\gamma_\phi \rho_{10}$$

Summing both contributions:
$$\frac{d\rho_{10}}{dt} = -\left( \frac{\Gamma}{2} + \gamma_\phi \right) \rho_{10} \implies \rho_{10}(t) = \rho_{10}(0) \exp\left(-\left[\frac{\Gamma}{2} + \gamma_\phi\right] t\right)$$
Matching exponents yields the familiar dephasing time:
$$\frac{1}{T_2} = \frac{\Gamma}{2} + \gamma_\phi = \frac{1}{2T_1} + \gamma_\phi$$
</details>

---

### Challenge 2: Numerical Verification of Coherence Decay Rate
For $\Gamma = 1.0$ and $\gamma_\phi = 0.5$, simulate the decay of an initial superposition state $|\psi_0\rangle = \frac{|0\rangle + |1\rangle}{\sqrt{2}}$. Verify that the numerical decay rate of $|\rho_{10}(t)|$ matches the theoretical rate $\Gamma/2 + \gamma_\phi = 0.5 + 0.5 = 1.000000$ to $<0.01\%$ accuracy.


In [ ]:
# Numerical Verification of Challenge 2: Coherence Decay Rate Benchmark

gamma_rad = 1.0
gamma_pure = 0.5
rate_theory = 0.5 * gamma_rad + gamma_pure  # 1.000000

sm_op = np.array([[0, 0], [1, 0]], dtype=complex)
sz_op = np.array([[1, 0], [0, -1]], dtype=complex)

sim_decay = LindbladMasterSimulator(
    H=np.zeros((2, 2), dtype=complex),
    c_ops=[sm_op, sz_op],
    rates=[gamma_rad, 0.5 * gamma_pure]
)

# Start in equal superposition |+>
psi_init = np.array([1.0, 1.0], dtype=complex) / np.sqrt(2.0)
rho_init = np.outer(psi_init, psi_init.conj())

t_points = np.linspace(0.0, 3.0, 50)
states_ev = sim_decay.evolve(rho_init, t_points)

# Extract off-diagonal coherences |rho_10(t)|
coherences = np.array([abs(s[1, 0]) for s in states_ev])

# Fit decay rate via log-linear regression: ln(|rho_10(t)|) = ln(0.5) - rate * t
log_coh = np.log(coherences)
slope, intercept = np.polyfit(t_points, log_coh, 1)
rate_numerical = -slope

print(f"--- Coherence Decay Rate Benchmark ---")
print(f"Theoretical Decay Rate (Gamma/2 + gamma_phi): {rate_theory:.6f}")
print(f"Numerical Fitted Decay Rate:                  {rate_numerical:.6f}")
print(f"Relative Discrepancy:                         {abs(rate_numerical - rate_theory) / rate_theory * 100:.4f}%")

assert abs(rate_numerical - rate_theory) < 1e-4, "Coherence decay rate check failed!"
print("Verification Passed: Numerical coherence decay matches theoretical GKSL formula to < 0.01% accuracy!")


---

## 5. Summary & Key Formulas

| Concept | Mathematical Expression | Physical Interpretation |
| :--- | :--- | :--- |
| **Lindblad Master Equation** | $\frac{d\hat{\rho}}{dt} = -i[\hat{H}, \hat{\rho}] + \sum_k \gamma_k \mathcal{D}[\hat{L}_k]\hat{\rho}$ | General Markovian completely positive open evolution |
| **Lindblad Dissipator** | $\mathcal{D}[\hat{L}]\hat{\rho} = \hat{L}\hat{\rho}\hat{L}^\dagger - \frac{1}{2}\{\hat{L}^\dagger\hat{L}, \hat{\rho}\}$ | Quantum jump term + non-Hermitian damping |
| **Vectorized Liouvillian** | $\hat{\mathcal{L}} = -i(\hat{I}\otimes\hat{H} - \hat{H}^T\otimes\hat{I}) + \dots$ | Supermatrix of dimension $d^2 \times d^2$ |
| **Steady State** | $\hat{\mathcal{L}} |\rho_{\text{ss}}\rangle\rangle = 0$ | Null eigenvector of Liouvillian supermatrix |
| **Dissipative Gap** | $\Delta_{\text{gap}} = \min_{j \ne 0} |\mathrm{Re}(\lambda_j)|$ | Governs asymptotic relaxation rate $\sim e^{-\Delta t}$ |

---

### Further Reading & Foundational Literature
* **G. Lindblad**, *On the Generators of Quantum Dynamical Semigroups*, Commun. Math. Phys. 48, 119 (1976).
* **V. Gorini, A. Kossakowski, & E. C. G. Sudarshan**, *Completely Positive Dynamical Semigroups of N-Level Systems*, J. Math. Phys. 17, 821 (1976).
* **H. P. Breuer & F. Petruccione**, *The Theory of Open Quantum Systems*, Oxford University Press (2002), Chapter 3.

---

### Next Notebook in Module 4
In **Notebook 4.3: Cavity Decay, Photon Loss & Thermal Reservoirs**, we apply the Lindblad formalism to continuous-variable optical modes:
* Cavity photon loss dissipator $\mathcal{D}[\sqrt{\kappa}\hat{a}]\hat{\rho}$.
* Phase-space Fokker-Planck equations for Wigner $W(x, p)$ and Glauber $P(\alpha)$ distributions.
* Coherent states shrinking under dissipation without loss of purity: $\alpha(t) = \alpha_0 e^{-\kappa t/2}$.
* Thermal noise injection and blackbody thermal bath equilibration ($n_{\text{th}} > 0$).
